In [42]:
# Imports
import os
from dotenv import load_dotenv
import requests
import json
import pandas as pd
import numpy as np

### 1) Directory Structure

In [43]:
load_dotenv()

api_key = os.getenv("API_KEY")
repo_owner = 'bedbad'
repo_name = 'justpyplot'

url = "https://api.github.com/graphql"
headers = {
    "Authorization": f"Bearer {api_key}"
}

# Rest of code in this cell is partly AI generated
# "... on Tree" is an Inline fragment. If the object is a Tree, it will fetch the entries.
def get_folder_contents(expression="HEAD:"):
    """Fetches the contents of a specific folder path."""
    query = """
    query($owner: String!, $name: String!, $expression: String!) {
      repository(owner: $owner, name: $name) {
        object(expression: $expression) {
          ... on Tree {
            entries {
              name
              path
              type
              extension
            }
          }
        }
      }
    }
    """
    variables = {
        "owner": repo_owner, 
        "name": repo_name, 
        "expression": expression
    }
    response = requests.post(url, json={'query': query, 'variables': variables}, headers=headers)
    return response.json()

all_items = []
# Start at the root of the repository
folders_to_check = ["HEAD:"]

while folders_to_check:
    current_folder = folders_to_check.pop(0)
    data = get_folder_contents(current_folder)
    try:
        entries = data['data']['repository']['object']['entries']
    except (KeyError, TypeError):
        continue
        
    for entry in entries:
        all_items.append({
            'Name': entry['name'],
            'Path': entry['path'],
            'Type': entry['type'], # 'blob' for files, 'tree' for folders
            'Extension': entry.get('extension', '')
        })
        
        # If the item is a folder (tree), queue it up to fetch its contents next
        if entry['type'] == 'tree':
            folders_to_check.append(f"HEAD:{entry['path']}/")

# Dump a sample to json for reference
with open("directory_query.json", "w") as file:
    json.dump(all_items[:100], file, indent=4)
    
print(f"Found {len(all_items)} total files and folders.")

Found 38 total files and folders.


In [44]:
df = pd.DataFrame(all_items)

# Change type column to something human readible
df['Type'] = np.where(df['Type'] == 'tree', 'folder', 'file')

# Sort alphabetically by path
df_folders = df.sort_values(by='Path').reset_index(drop=True)

# Create CSV file
df_folders.to_csv("repo_directory_structure.csv", index=False)

print("Created CSV File")
df_folders.head(5)

Created CSV File


,Name,Path,Type,Extension
0,.github,.github,folder,
1,funding.yaml,.github/funding.yaml,file,.yaml
2,workflows,.github/workflows,folder,
3,python-publish.yml,.github/workflows/python-publish.yml,file,.yml
4,.gitignore,.gitignore,file,


### 2) Naming Conventions

In [45]:
def get_file_contents(expression="HEAD:"):
    """Fetches the contents of a specific folder path."""
    query = """
    query($owner: String!, $name: String!, $expression: String!) {
      repository(owner: $owner, name: $name) {
        object(expression: $expression) {
          ... on Blob {
              text
            }
        }
      }
    }
    """
    variables = {
        "owner": repo_owner, 
        "name": repo_name, 
        "expression": expression
    }
    response = requests.post(url, json={'query': query, 'variables': variables}, headers=headers)
    return response.json()

# We're only extracting Python repos. So get all .py files
df_files = df.loc[df['Extension'] == '.py', ['Path']]

# Prefix with HEAD: for the query
df_files['Path'] = 'HEAD:' + df_files['Path']
files_to_check = df_files.squeeze().tolist()

print(files_to_check)
all_items = []

while files_to_check:
    current_file = files_to_check.pop()
    data = get_file_contents(current_file)
    try:
        raw_text = data['data']['repository']['object']['text']
    except (KeyError, TypeError):
        continue
    
    # Remove the prefix HEAD: from the query
    current_file = current_file[5:]
    all_items.append({
        'Path': current_file,
        'Raw text': raw_text
    })

# Dump a sample to json for reference
with open("file_query.json", "w") as file:
    json.dump(all_items[:100], file, indent=4)
    
print(f"Found {len(all_items)} total Python files.")

['HEAD:docs/conf.py', 'HEAD:examples/__init__.py', 'HEAD:justpyplot/__init__.py', 'HEAD:justpyplot/justpyplot.py', 'HEAD:justpyplot/textrender.py', 'HEAD:tests/__init__.py', 'HEAD:tests/test.py', 'HEAD:tests/test_basic.py', 'HEAD:tests/test_basic_plot.py', 'HEAD:tests/test_plot.py', 'HEAD:tests/test_plot_components.py', 'HEAD:tests/test_standalone.py', 'HEAD:tests/test_textrender.py', 'HEAD:examples/mug_objectron/__init__.py', 'HEAD:examples/mug_objectron/demo.py']
Found 15 total Python files.


In [48]:
# Migrate to Pandas Dataframe
df_files = pd.DataFrame(all_items)

# Merge with the original folders dataframe from Section 1.
df_folder_and_files = pd.merge(df_folders, df_files, on='Path')

# Final dataframe for Section 2.
df_folder_and_files.head(5)

,Name,Path,Type,Extension,Raw text
0,conf.py,docs/conf.py,file,.py,import os\nimport sys\n\n# Add the project roo...
1,__init__.py,examples/__init__.py,file,.py,
2,__init__.py,examples/mug_objectron/__init__.py,file,.py,
3,demo.py,examples/mug_objectron/demo.py,file,.py,import cv2\nimport mediapipe as mp\nimport num...
4,__init__.py,justpyplot/__init__.py,file,.py,VERSION = '0.1.0'\n
